In [0]:
import os
import sys
import importlib

src_path = os.path.abspath("../src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

import gold
importlib.reload(gold)

from config import CATALOG, SCHEMA

print("Silver source:", f"{CATALOG}.{SCHEMA}.silver_transactions")
print("Gold tables:")
print(gold.GOLD_DAILY_TABLE)
print(gold.GOLD_PRODUCT_TABLE)
print(gold.GOLD_STORE_TABLE)
print(gold.GOLD_PAYMENT_TABLE)

In [0]:
gold.run_gold(spark)

print("Gold pipeline executed.")

In [0]:
daily_df = spark.table(gold.GOLD_DAILY_TABLE)
product_df = spark.table(gold.GOLD_PRODUCT_TABLE)
store_df = spark.table(gold.GOLD_STORE_TABLE)
payment_df = spark.table(gold.GOLD_PAYMENT_TABLE)

print("Daily sales schema:")
daily_df.printSchema()
display(daily_df.orderBy("transaction_date"))

print("Product performance:")
display(
    product_df
    .orderBy(product_df["sales_amount"].desc())
    .limit(10)
)

print("Store performance:")
display(
    store_df
    .orderBy(store_df["sales_amount"].desc())
)

print("Payment performance:")
display(
    payment_df
    .orderBy(payment_df["sales_amount"].desc())
)


In [0]:
reconciliation_passed = gold.reconcile(spark)

if not reconciliation_passed:
    raise ValueError("Gold reconciliation failed.")

print("Gold reconciliation passed.")


In [0]:
idempotency_passed = gold.check_idempotency(spark)

if not idempotency_passed:
    raise ValueError("Gold idempotency check failed.")

print("Gold pipeline completed successfully.")
